In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt

# Path
path_runcing = '/content/drive/MyDrive/sukulen/daun_runcing'
path_bulat = '/content/drive/MyDrive/sukulen/daun_bulat'

# Folder hasil
path_output = '/content/drive/MyDrive/sukulen_cleaned'
folders = ['daun_runcing', 'daun_bulat']

# folder output
for f in folders:
    os.makedirs(os.path.join(path_output, f), exist_ok=True)

def preprocess_image(image_path, target_size=(224, 224)):
    # BACA GAMBAR
    img = cv2.imread(image_path)
    if img is None:
        return None

    # CENTER CROP
    h, w = img.shape[:2]
    side = min(h, w)
    crop_size = int(side * 0.7) # Ambil 70% area tengah
    start_x = (w - crop_size) // 2
    start_y = (h - crop_size) // 2
    img = img[start_y:start_y+crop_size, start_x:start_x+crop_size]

    # GRAYSCALE
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # RESIZE
    img = cv2.resize(img, target_size)

    # NORMALISASI PIKSEL 0-1
    img_normalized = img.astype('float32') / 255.0

    return img_normalized

# EKSEKUSI
print("Memulai Preprocessing...")

for folder in folders:
    current_path = os.path.join('/content/drive/MyDrive/sukulen', folder)
    save_path = os.path.join(path_output, folder)

    files = [f for f in os.listdir(current_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for filename in files:
        img_path = os.path.join(current_path, filename)
        processed_img = preprocess_image(img_path)

        if processed_img is not None:
            # Simpan kembali sebagai gambar
            final_to_save = (processed_img * 255).astype(np.uint8)
            cv2.imwrite(os.path.join(save_path, filename), final_to_save)

print(f"Selesai! Hasil pembersihan ada di: {path_output}")

Memulai Preprocessing...
Selesai! Hasil pembersihan ada di: /content/drive/MyDrive/sukulen_cleaned


In [3]:
import cv2
import os
import random
import numpy as np

# Path hasil cleaning
path_cleaned = '/content/drive/MyDrive/sukulen_cleaned'
folders = ['daun_runcing', 'daun_bulat']
target_jumlah = 200

def augmentasi_variasi(img):
    if img is None:
        return None

    pilihan = random.choice(['flip', 'rotasi', 'cerah'])
    if pilihan == 'flip':
        return cv2.flip(img, 1)
    elif pilihan == 'rotasi':
        angle = random.randint(-15, 15)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h))
    else:
        value = random.uniform(0.8, 1.2)
        # Operasi perkalian dilakukan pada array numpy
        return np.clip(img * value, 0, 255).astype(np.uint8)

print("Memulai Standarisasi Nama dan Augmentasi...")

for folder in folders:
    p = os.path.join(path_cleaned, folder)

    # Standarisasi Nama File Asli
    files_awal = sorted([f for f in os.listdir(p) if f.endswith(('.jpg', '.png', '.jpeg'))])

    print(f"Mengubah nama asli di kelas {folder}...")
    temp_files_asli = []
    for idx, nama_lama in enumerate(files_awal):
        # Menghindari mengubah nama jika file sudah dalam format standar (ID.0.jpg)
        if not nama_lama.endswith('.0.jpg'):
            nama_standar = f"{idx + 1}.0.jpg"
            path_lama = os.path.join(p, nama_lama)
            path_baru = os.path.join(p, nama_standar)
            os.rename(path_lama, path_baru)
            temp_files_asli.append(nama_standar)
        else:
            temp_files_asli.append(nama_lama)

    # Augmentasi hingga mencapai target 200
    # Menghitung jumlah file saat ini di folder
    files_sekarang = [f for f in os.listdir(p) if f.endswith(('.jpg', '.png', '.jpeg'))]
    jumlah_saat_ini = len(files_sekarang)

    # Inisialisasi tracker untuk setiap file asli
    tracker_varian = {f: 0 for f in temp_files_asli}

    print(f"Kelas {folder}: Saat ini ada {jumlah_saat_ini} foto. Menambah {target_jumlah - jumlah_saat_ini} foto...")

    while jumlah_saat_ini < target_jumlah:
        # Pilih foto secara acak
        file_induk = random.choice(temp_files_asli)
        id_tanaman = file_induk.split('.')[0]

        path_file = os.path.join(p, file_induk)
        img = cv2.imread(path_file, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            img_aug = augmentasi_variasi(img)

            if img_aug is not None:
                tracker_varian[file_induk] += 1
                nama_aug = f"{id_tanaman}.{tracker_varian[file_induk]}.jpg"

                while os.path.exists(os.path.join(p, nama_aug)):
                    tracker_varian[file_induk] += 1
                    nama_aug = f"{id_tanaman}.{tracker_varian[file_induk]}.jpg"

                cv2.imwrite(os.path.join(p, nama_aug), img_aug)
                jumlah_saat_ini += 1
        else:
            print(f"Peringatan: Gagal membaca file {path_file}. Lewati...")
            continue

print("Selesai! Dataset sekarang standar (ID.Varian) dan berjumlah 200 per kelas.")

Memulai Standarisasi Nama dan Augmentasi...
Mengubah nama asli di kelas daun_runcing...
Kelas daun_runcing: Saat ini ada 148 foto. Menambah 52 foto...
Mengubah nama asli di kelas daun_bulat...
Kelas daun_bulat: Saat ini ada 116 foto. Menambah 84 foto...
Selesai! Dataset sekarang standar (ID.Varian) dan berjumlah 200 per kelas.


In [4]:
import cv2
import numpy as np
import os

# Mendefinisikan Path folder hasil augmentasi
path_base = '/content/drive/MyDrive/sukulen_cleaned'
folders = ['daun_runcing', 'daun_bulat']

data = []
labels = []

print("Proses konversi gambar ke NumPy Array sedang berjalan...")

for folder in folders:
    path = os.path.join(path_base, folder)
    # Mapping Label: 0 untuk Runcing, 1 untuk Bulat
    label = 0 if folder == 'daun_runcing' else 1

    # Mengambil semua file gambar
    files = [f for f in os.listdir(path) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for img_name in files:
        img_path = os.path.join(path, img_name)
        # Membaca gambar dalam mode Grayscale
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            # Flattening, yaitu engubah matriks 2D (misal 224x224) menjadi 1D array panjang
            # Bisa untuk algoritma tradisional, semacam SVM atau KNN
            img_flat = img.flatten()
            data.append(img_flat)
            labels.append(label)

# Mengubah ke format NumPy Array dan Normalisasi (0-1)
X = np.array(data, dtype='float32') / 255.0
y = np.array(labels)

# Menyiimpan File ke Google Drive
np.save('/content/drive/MyDrive/X_sukulen.npy', X)
np.save('/content/drive/MyDrive/y_sukulen.npy', y)

print("--- EKSEKUSI BERHASIL ---")
print(f"Total Data (X): {X.shape[0]} sampel dengan {X.shape[1]} fitur piksel.")
print(f"Total Label (y): {y.shape[0]} label tersimpan.")
print("File X_sukulen.npy dan y_sukulen.npy siap digunakan.")

Proses konversi gambar ke NumPy Array sedang berjalan...
--- EKSEKUSI BERHASIL ---
Total Data (X): 400 sampel dengan 50176 fitur piksel.
Total Label (y): 400 label tersimpan.
File X_sukulen.npy dan y_sukulen.npy siap digunakan.
